## Cell 65: report helpers
**INSERT AFTER:** the code cell of **Cell 1** (`## Cell 1: metric report`, it defines `report`)

In [ ]:
# 10_protocol_ladder - Cell 65: report helpers - insert right after Cell 1 - for the figures and tables of the results that have none yet (read-only). Uses the notebook's report() from Cell 1, so every new figure has the format of
# the existing ones (ROC of the models with the trivial baselines dashed; confusion matrices summed over seeds; table = mean over seeds (min-max)). Run once per kernel, before Cells 66-70.
import contextlib
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve
from circmtl import ladder as L
assert all(n in globals() for n in ("report", "gd", "D", "OUT", "OLD", "np", "pd")), "run Cell 0 and Cell 1 first (they define report, OUT, OLD and gd)"
assert callable(report), "report has been overwritten; rerun Cell 1"
ROUND1 = D / "train_results_round1"
BL_HOST = ("disease_degree", "circrna_degree", "log_degree_sum")                                                                   # as in the S0 and B reports
BL_PAIR = ("disease_degree", "circrna_degree", "circrna_inverse_degree", "log_degree_sum", "kernel_circ", "kernel_dis", "kernel_both", "shortcut_logit")      # as in Cells 13, 16 and 19
nC65, nD65 = gd.N["circRNA"], gd.N["disease"]

def _bs_strict(d, kind):
    """popularity baselines of the host-gene folds, from the fold's TRAINING pairs only (as Cell 1)"""
    out = np.zeros(len(d))
    for (s, k), ix in d.groupby(["seed", "fold"]).indices.items():
        tr = gd.load_fold(int(s), int(k))["disease_train"]; dd = np.bincount(tr[:, 1], minlength=nD65); cd = np.bincount(tr[:, 0], minlength=nC65)
        c, t = d["c"].to_numpy()[ix], d["t"].to_numpy()[ix]
        out[ix] = {"disease_degree": dd[t], "circrna_degree": cd[c], "log_degree_sum": np.log1p(dd[t]) + np.log1p(cd[c])}[kind]
    return out

def _bs_pair_factory(tag):
    def _bs_pair(d, kind):                                                           # as Cell 13: baselines of the pair-level folds from TRAINING pairs only
        out = np.zeros(len(d))
        for (s, k), ix in d.groupby(["seed", "fold"]).indices.items():
            fd = L.pair_level_fold(gd, int(s), int(k)); Y = L.label_matrix(fd["disease_train"], nC65, nD65); dd, dc = Y.sum(0), Y.sum(1)
            c, t = d["c"].to_numpy()[ix], d["t"].to_numpy()[ix]
            if kind == "disease_degree": sc = dd[t]
            elif kind == "circrna_degree": sc = dc[c]
            elif kind == "circrna_inverse_degree": sc = -dc[c]
            elif kind == "log_degree_sum": sc = np.log1p(dd[t]) + np.log1p(dc[c])
            elif kind.startswith("kernel_"): sc = L.kernel_scores(Y)[kind][c, t]
            elif kind == "shortcut_logit":                                           # fitted on validation only
                z = np.load(OUT / "frozen_pair" / f"neg{tag}_seed{int(s)}_fold{int(k)}.npz")
                pv = np.vstack([fd["disease_val"], z["neg_val"]]); yv_ = np.r_[np.ones(len(fd["disease_val"])), np.zeros(len(z["neg_val"]))]
                clf = LogisticRegression(C=1.0).fit(np.c_[np.log1p(dd[pv[:, 1]]), np.log1p(dc[pv[:, 0]])], yv_)
                sc = clf.decision_function(np.c_[np.log1p(dd[t]), np.log1p(dc[c])])
            else: raise ValueError(kind)
            out[ix] = sc
        return out
    return _bs_pair

def _bs_external(d, kind):
    """run X: new circRNAs have no training pairs; the popularity baseline uses the dataset_v1 degrees of the diseases (as Cell 49)"""
    dd = np.bincount(gd.E["disease"][:, 1], minlength=nD65).astype(float)[d["t"].to_numpy()]
    return {"disease_degree": dd, "circrna_degree": np.zeros(len(d)), "log_degree_sum": np.log1p(dd)}[kind]

@contextlib.contextmanager
def baselines_for(kind, tag=None):
    """report() looks up the global name baseline_scores; swap it for the right protocol and ALWAYS restore it"""
    keep = globals()["baseline_scores"] if "baseline_scores" in globals() else None
    globals()["baseline_scores"] = {"strict": _bs_strict, "pair": _bs_pair_factory(tag) if kind == "pair" else None, "external": _bs_external}[kind]
    try: yield
    finally:
        if keep is not None: globals()["baseline_scores"] = keep

def read_arm(folder, model, label=None, pattern=None):
    """all prediction files of one model in one folder; the model column is relabelled when a label is given"""
    files = sorted(Path(folder).glob(pattern or f"pred_{model}_seed*.csv")); assert files, f"no prediction files for {model} in {folder}"
    P = pd.concat([pd.read_csv(f) for f in files], ignore_index=True); P = P[P["model"] == model].copy(); assert len(P), f"model {model} not in the files of {folder}"
    if label: P["model"] = label
    return P

def run_report(P, models, tag, scheme, kind, ptag=None, baselines=None):
    """the notebook's report() with the right baselines; handles files without a fold column (run X) or without score_rank"""
    P = P.copy()
    if "fold" not in P: P["fold"] = 0
    if "score_rank" not in P: P["score_rank"] = P.groupby(["model", "seed", "fold", "task", "scheme"])["score"].rank(pct=True)
    if "threshold" in P:
        has = P.groupby("model")["threshold"].apply(lambda s: bool(s.notna().all())); assert has.all() or not has.any(), "some models have thresholds and others not: do not mix them in one figure"
    with baselines_for(kind, ptag):
        out = report(P, list(models), tag=tag, scheme=scheme, baselines=baselines or (BL_PAIR if kind == "pair" else BL_HOST))
    if kind == "pair": by_circrna_status(P[(P["task"] == "disease") & (P["scheme"] == scheme)], list(models), ptag, scheme)
    return out

def read_set(folder, models, **kw):
    return pd.concat([read_arm(folder, m, **kw) for m in models], ignore_index=True)

def by_circrna_status(P, models, run, scheme):
    """AUROC split by whether the pair's circRNA has >= 1 training pair (as Cell 13); pair-level protocols only"""
    bs = _bs_pair_factory(run); P = P[P["model"].isin(models)].reset_index(drop=True)
    warm = np.zeros(len(P), bool)
    for (s, k), ix in P.groupby(["seed", "fold"]).indices.items():
        dc = np.bincount(L.pair_level_fold(gd, int(s), int(k))["disease_train"][:, 0], minlength=nC65)
        warm[ix] = dc[P["c"].to_numpy()[ix]] > 0
    msk_b = (P["model"] == models[0]).to_numpy(); base = P[msk_b][["seed", "fold", "c", "t", "y"]].reset_index(drop=True); warm_b = warm[msk_b]
    comp = {n: bs(base, n) for n in ("disease_degree", "shortcut_logit")}
    rows = []
    for name, flag in (("warm (circRNA has training pairs)", True), ("cold (circRNA has none)", False)):
        sel = warm_b == flag; r = {"stratum": name, "pos": int(base["y"][sel].sum()), "neg": int((1 - base["y"][sel]).sum())}
        for m in models:
            g = P[(P["model"] == m).to_numpy() & (warm == flag)]
            a = [roc_auc_score(x["y"], x["score_rank"]) for _, x in g.groupby("seed") if 0 < x["y"].sum() < len(x)]; r[m] = f"{np.mean(a):.3f}" if a else "n/a"
        for cn, v in comp.items():
            a = []
            for sd in sorted(base["seed"].unique()):
                mk = sel & (base["seed"] == sd).to_numpy()
                if 0 < base["y"][mk].sum() < mk.sum(): a.append(roc_auc_score(base["y"][mk], v[mk]))
            r[cn] = f"{np.mean(a):.3f}" if a else "n/a"
        rows.append(r)
    print(f"\nAUROC by circRNA status [{run}, {scheme}] (mean over seeds):"); print(pd.DataFrame(rows).to_string(index=False))

SCHEME65 = {"A": "degree_matched", "C": "uniform_pair", "C2": "uniform_test_circ"}
def report_arm(protocol, arm):
    """enhancement arm (depth or sim) on one protocol: S0 (both negative schemes), A or C2; M1, M2, M3"""
    folder = ROUND1 / f"pred_{protocol}_{arm}"; assert (folder / "DONE.txt").exists(), f"{protocol} {arm} is not complete in {folder}"
    P = read_set(folder, ("M1", "M2", "M3")); models = ["M1", "M2", "M3"]
    for scheme in (("degree_matched", "random") if protocol == "S0" else (SCHEME65[protocol],)):
        print(); run_report(P, models, tag=f"{protocol}_{arm}", scheme=scheme, kind="strict" if protocol == "S0" else "pair", ptag=None if protocol == "S0" else protocol)

def report_shuffled(run):
    """shuffled-auxiliary control: M3 against M3_shuf on A, C or C2 (a control, not a result)"""
    ctl = OUT / f"pred_{run}_M3shuf"; assert (ctl / "DONE.txt").exists(), f"the control for {run} is not complete in {ctl}"
    P = pd.concat([read_arm(OUT / f"pred_{run}", "M3"), read_arm(ctl, "M3_shuf", pattern="pred_M3shuf_seed*_fold*.csv")], ignore_index=True)
    print(); run_report(P, ["M3", "M3_shuf"], tag=f"{run}_shuffled_control", scheme=SCHEME65[run], kind="pair", ptag=run)

print("Cell 65 ok: helpers defined (read_arm, run_report, baselines_for); figures and tables go to", OUT, "as report_<tag>_<scheme>.png and .csv")


## Cell 68a: shuffled-auxiliary control, run A - figure and table
**INSERT AFTER:** the code cell under the markdown **`## Cell 23: the comparison A`**

In [ ]:
# 10_protocol_ladder - Cell 68a: shuffled-auxiliary control, run A (M3 against M3_shuf) - figure and table; needs Cell 65
report_shuffled("A")


## Cell 68b: shuffled-auxiliary control, run C - figure and table
**INSERT AFTER:** the code cell under the markdown **`## Cell 23: the comparison C`**

In [ ]:
# 10_protocol_ladder - Cell 68b: shuffled-auxiliary control, run C - figure and table; needs Cell 65
report_shuffled("C")


## Cell 68c: shuffled-auxiliary control, run C2 - figure and table
**INSERT AFTER:** the code cell under the markdown **`## Cell 23: the comparison C2`**

In [ ]:
# 10_protocol_ladder - Cell 68c: shuffled-auxiliary control, run C2 - figure and table; needs Cell 65
report_shuffled("C2")


## Cell 67a: enhancement arm sim, protocol S0 - figure and table
**INSERT AFTER:** the code cell under **`## Cell 34: compare arm and main`** (arm sim, S0)

In [ ]:
# 10_protocol_ladder - Cell 67a: enhancement round 1, arm sim, protocol S0 - figure and table; needs Cell 65
report_arm("S0", "sim")


## Cell 67b: enhancement arm depth, protocol S0 - figure and table
**INSERT AFTER:** the code cell under **`## Cell 36:  depth S0`**

In [ ]:
# 10_protocol_ladder - Cell 67b: enhancement round 1, arm depth, protocol S0 - figure and table; needs Cell 65
report_arm("S0", "depth")


## Cell 67c: enhancement arm depth, protocol A - figure and table
**INSERT AFTER:** the code cell under **`## Cell 36:  depth A`**

In [ ]:
# 10_protocol_ladder - Cell 67c: enhancement round 1, arm depth, protocol A - figure and table; needs Cell 65
report_arm("A", "depth")


## Cell 67d: enhancement arm sim, protocol A - figure and table
**INSERT AFTER:** the code cell under **`## Cell 36:  sim  A`**

In [ ]:
# 10_protocol_ladder - Cell 67d: enhancement round 1, arm sim, protocol A - figure and table; needs Cell 65
report_arm("A", "sim")


## Cell 67e: enhancement arm depth, protocol C2 - figure and table
**INSERT AFTER:** the code cell under **`## Cell 36:  depth C2`**

In [ ]:
# 10_protocol_ladder - Cell 67e: enhancement round 1, arm depth, protocol C2 - figure and table; needs Cell 65
report_arm("C2", "depth")


## Cell 67f: enhancement arm sim, protocol C2 - figure and table
**INSERT AFTER:** the code cell under **`## Cell 36:  sim  C2`**

In [ ]:
# 10_protocol_ladder - Cell 67f: enhancement round 1, arm sim, protocol C2 - figure and table; needs Cell 65
report_arm("C2", "sim")


## Cell 66: run X - figures and tables
**INSERT AFTER:** the code cell under **`## Cell 45:`** (run X results)

In [ ]:
# 10_protocol_ladder - Cell 66: run X, figures and tables (primary = degree-matched, secondary = uniform negatives; 2-layer and 3-layer models); needs Cell 65
XD = ROUND1 / "pred_X"
assert (XD / "DONE.txt").exists(), "run X is not complete"
for scheme, name in (("degree_matched", "X_primary"), ("uniform_pair", "X_secondary")):
    for models, suffix in ((["M1", "M2", "M3"], ""), (["M1d", "M2d", "M3d"], "_3layer")):
        P = pd.concat([read_arm(XD, m) for m in models], ignore_index=True)
        print()
        run_report(P, models, tag=name + suffix, scheme=scheme, kind="external")


## Cell 70: MLP extension - figures and tables
**INSERT AFTER:** the code cell under **`## Cell 62:`** (MLP extension analysis)

In [ ]:
# 10_protocol_ladder - Cell 70: MLP decoder extension, figures and tables (read-only): DistMult against MLP-cat and MLP-catprod under S0 (both negative schemes), A, C and C2, one figure per model (M1, M3).
# DistMult = the frozen run of each protocol; MLP = the folders of Cell 61. Needs Cell 65.
assert "run_report" in globals(), "run Cell 65 first"
REF70 = {"S0": OLD, "A": OUT / "pred_A", "C": OUT / "pred_C", "C2": OUT / "pred_C2"}
PROT70 = (("S0", "strict", None, ("degree_matched", "random")), ("A", "pair", "A", ("degree_matched",)), ("C", "pair", "C", ("uniform_pair",)), ("C2", "pair", "C2", ("uniform_test_circ",)))
for protocol, kind, ptag, schemes in PROT70:
    srcs = {"DistMult": REF70[protocol]}
    for lab, dec in (("MLP-cat", "cat"), ("MLP-catprod", "catprod")):
        f = ROUND1 / f"pred_{protocol}_mlp{dec}"
        if (f / "DONE.txt").exists(): srcs[lab] = f
        else: print(f"[{protocol}] {lab} not complete in {f} (left out)")
    if len(srcs) == 1: continue
    for m in ("M1", "M3"):
        P = pd.concat([read_arm(f, m, label=lab) for lab, f in srcs.items()], ignore_index=True)
        for scheme in schemes:
            print()
            run_report(P, list(srcs), tag=f"{protocol}_MLP_{m}", scheme=scheme, kind=kind, ptag=ptag)


## Cell 69: decoder arms on S0 - figures and tables
**INSERT AFTER:** the code cell under **`## Cell 64:`** (MLP-b analysis)

In [ ]:
# 10_protocol_ladder - Cell 69: decoder arms on S0, figures and tables (read-only): DM-b (DistMult, random-disease training negatives), MLP-b (MLP decoder, same negatives) and MLPx-b (MLP with the
# product input), one figure per model (M1, M3) and per test negative scheme. The S0 figure of the main models (DM-a: degree-matched training negatives) is the one of Cell 1. Needs Cell 65.
assert "run_report" in globals(), "run Cell 65 first"
SRC69 = {"DM-b": ROUND1 / "pred_S0_DMb", "MLP-b": ROUND1 / "pred_S0_MLPb_cat", "MLPx-b": ROUND1 / "pred_S0_MLPb_catprod"}
SRC69 = {lab: f for lab, f in SRC69.items() if (f / "DONE.txt").exists()}
print("arms found:", list(SRC69))
for m in ("M1", "M3"):
    for scheme in ("degree_matched", "random"):
        P = pd.concat([read_arm(f, m, label=lab) for lab, f in SRC69.items()], ignore_index=True)
        print()
        run_report(P, list(SRC69), tag=f"S0_decoder_arms_{m}", scheme=scheme, kind="strict")


## Cell 71: per-circRNA ranking on S0 - figure and table
**INSERT AFTER:** the cell of **Cell 69** just inserted (the last new cell)

In [ ]:
# 10_protocol_ladder - Cell 71: per-circRNA ranking of the true disease among all diseases on S0 (read-only): cumulative hit rate against k for the popularity ranker, a random list and the arms
# DM-a, DM-b, MLP-b and MLPx-b (models M1 and M3), from the full score matrices; filtered rank (the circRNA's other known diseases removed). Figure + table + csv in OUT. Needs Cell 65.
assert "run_report" in globals(), "run Cell 65 first"
ARMS71 = {"DM-a": ROUND1 / "pred_S0rank", "DM-b": ROUND1 / "pred_S0_DMb", "MLP-b": ROUND1 / "pred_S0_MLPb_cat", "MLPx-b": ROUND1 / "pred_S0_MLPb_catprod"}
ARMS71 = {lab: f for lab, f in ARMS71.items() if (f / "DONE.txt").exists()}
K71, SEEDS71, n_dis71 = np.arange(1, 51), range(3), gd.N["disease"]
K_known71 = np.zeros((gd.N["circRNA"], n_dis71), bool); K_known71[gd.E["disease"][:, 0], gd.E["disease"][:, 1]] = True

def filtered_ranks(folder, m, s, popularity=False):
    out, ncand = [], []
    for k in range(5):
        z = np.load(folder / f"full_{m}_seed{s}_fold{k}.npz")
        rows = pd.Series(np.arange(len(z["test_c"])), index=z["test_c"]).loc[z["pos_c"]].to_numpy()
        Sg = np.tile(z["train_degree"].astype(float), (len(rows), 1)) if popularity else z["scores"][rows]
        st = Sg[np.arange(len(rows)), z["pos_t"]]; other = K_known71[z["pos_c"]].copy(); other[np.arange(len(rows)), z["pos_t"]] = False
        out.append(1 + ((Sg > st[:, None]) & ~other).sum(1) + 0.5 * (((Sg == st[:, None]) & ~other).sum(1) - 1)); ncand.append(n_dis71 - other.sum(1))
    return np.concatenate(out), np.concatenate(ncand)

def curves(rank_by_seed, ncand_by_seed):
    hit = np.mean([[np.mean(r <= k) for k in K71] for r in rank_by_seed], axis=0)
    mrr = np.mean([np.mean(1.0 / r) for r in rank_by_seed]); med = np.mean([np.median(r) for r in rank_by_seed]); return hit, mrr, med
first = next(iter(ARMS71.values()))
pop = [filtered_ranks(first, "M1", s, popularity=True) for s in SEEDS71]
rand_hit = np.mean([[np.mean(np.minimum(k, nc) / nc) for k in K71] for _, nc in pop], axis=0)
rand_mrr = np.mean([np.mean([np.sum(1.0 / np.arange(1, n + 1)) / n for n in np.unique(nc)]) for _, nc in pop])
pop_hit, pop_mrr, pop_med = curves([r for r, _ in pop], [n for _, n in pop])
rows, fig_data = [{"ranker": "popularity (training degree)", "MRR": pop_mrr, "median rank": pop_med, **{f"hit@{k}": pop_hit[k - 1] for k in (1, 5, 10, 20, 50)}},
                  {"ranker": "random list (expected)", "MRR": rand_mrr, "median rank": np.nan, **{f"hit@{k}": rand_hit[k - 1] for k in (1, 5, 10, 20, 50)}}], {}
for m in ("M1", "M3"):
    for lab, f in ARMS71.items():
        rk = [filtered_ranks(f, m, s) for s in SEEDS71]; hit, mrr, med = curves([r for r, _ in rk], [n for _, n in rk]); fig_data[(m, lab)] = hit
        rows.append({"ranker": f"{lab} {m}", "MRR": mrr, "median rank": med, **{f"hit@{k}": hit[k - 1] for k in (1, 5, 10, 20, 50)}})
T71 = pd.DataFrame(rows).set_index("ranker"); print(f"per-circRNA ranking on S0 (mean over {len(list(SEEDS71))} seeds; filtered rank among {n_dis71} diseases)"); print(T71.round(4).to_string())
T71.to_csv(OUT / "report_S0_ranking.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, m in zip(axes, ("M1", "M3")):
    for lab in ARMS71: ax.plot(K71, fig_data[(m, lab)], label=f"{lab} ({T71.loc[f'{lab} {m}', 'hit@10']:.3f} at k=10)")
    ax.plot(K71, pop_hit, color="black", ls="--", label=f"popularity ({pop_hit[9]:.3f})"); ax.plot(K71, rand_hit, color="grey", ls=":", label=f"random list ({rand_hit[9]:.3f})")
    ax.set(xlabel="k (top-k diseases of the circRNA)", title=f"{m}: true disease within the top k [S0]"); ax.legend(fontsize=7, loc="lower right")
axes[0].set_ylabel("fraction of test positives"); plt.tight_layout(); plt.savefig(OUT / "report_S0_ranking.png", dpi=200); plt.show()
